<div dir="rtl">

# 🧪 مختبر اختبار مستودعات المتجهات (Vector Stores Testing Workbench)

هذا الكراس مخصص ليكون **بيئة اختبار وتجربة عملية متكاملة** لجميع أنواع الـ **Vector Stores** في LangChain، مع **تثبيت وتوحيد المراحل السابقة** (التحميل، التقسيم الذكي، والتضمين المحلي).

---

### 📋 خريطة خطوات الاختبار في هذا الكراس:
- **المرحلة 1 (تثبيت التحميل)**: قراءة `data/sample.txt` باستخدام `TextLoader`.
- **المرحلة 2 (تثبيت التقسيم)**: تقطيع النص باستخدام `RecursiveCharacterTextSplitter`.
- **المرحلة 3 (تثبيت التضمين)**: تهيئة نموذج التضمين القياسي `sentence-transformers/all-MiniLM-L6-v2`.
- **المرحلة 4 (اختبار مستودعات المتجهات)**:
  1. `InMemoryVectorStore` (LangChain Core)
  2. `FAISS` (Facebook AI Similarity Search - In-Memory & Persistence)
  3. `SKLearnVectorStore` (Scikit-Learn KNN & JSON Serializer)
  4. `DocArrayInMemorySearch` (Ultra-fast memory store)
  5. `Chroma` (LangChain Chroma DB)
  6. `QdrantVectorStore` (Qdrant Rust Engine)
- **المرحلة 5 (اختبار المُسترجعات وأنماط البحث)**: مقارنة `Similarity` و `MMR` و `Threshold`.
- **المرحلة 6 (سلسلة الـ RAG الكاملة)**: دمج المستودع مع LCEL و Prompt لتوليد إجابات موثقة.

</div>


<div dir="rtl">

## 1️⃣ المرحلة الأولى والثانية: تحميل المستند وتقسيمه (Fixed Data & Splitter)

</div>


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. تحميل البيئة
load_dotenv(find_dotenv())

# 2. تحديد مسار ملف البيانات بدقة
DATA_DIR = Path("data") if Path("data").exists() else Path("../../data") if Path("../../data").exists() else Path("../data")
sample_file = DATA_DIR / "sample.txt"

# 3. قراءة المستند
loader = TextLoader(str(sample_file), encoding="utf-8")
raw_documents = loader.load()

# 4. تقسيم المستند إلى قطع مركزة
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

docs = text_splitter.split_documents(raw_documents)

# ترقيم الميتاداتا
for i, d in enumerate(docs, 1):
    d.metadata["chunk_id"] = i

print(f"✅ تم تحميل المستند: {len(raw_documents[0].page_content):,} حرف.")
print(f"✅ تم تقسيم المستند إلى: {len(docs)} قطعة.")
print(f"📊 عينة من القطعة الأولى:\n{docs[0].page_content[:120]}...")


<div dir="rtl">

## 2️⃣ المرحلة الثالثة: تثبيت نموذج التضمين (Fixed Embedding Model)
نستخدم الحزمة الحديثة `langchain_huggingface` مع نموذج `all-MiniLM-L6-v2` المجاني والمحلي 100%.

</div>


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# تهيئة نموذج التضمين القياسي
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ تم تجهيز نموذج التضمين بنجاح (384 Dimensions)!")


<div dir="rtl">

## 3️⃣ النوع الأول: `InMemoryVectorStore` (LangChain Core)
مستودع متجهات خفيف جداً مدمج في نواة لانج تشين ويعمل بالكامل في الذاكرة RAM.

</div>


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

# إنشاء المستودع وتضمين النصوص
in_memory_store = InMemoryVectorStore.from_documents(docs, hf_embeddings)

# استعلام تجريبي
query1 = "What is the recommended chunk_size and chunk_overlap for general Q&A?"
mem_results = in_memory_store.similarity_search(query1, k=2)

print("🎯 نتائج البحث في InMemoryVectorStore:")
for i, res in enumerate(mem_results, 1):
    print(f"{i}. [Chunk #{res.metadata.get('chunk_id')}]: {res.page_content[:130]}...
")


<div dir="rtl">

## 4️⃣ النوع الثاني: `FAISS` (Meta AI - In-Memory & Local Persistence)
أشهر وأسرع مكتبة للمتجهات الكثيفة، مع إمكانية الحفظ والاسترجاع من القرص.

</div>


In [ ]:
from langchain_community.vectorstores import FAISS

# 1. بناء الفهرس
faiss_store = FAISS.from_documents(docs, hf_embeddings)

# 2. البحث مع حساب درجات المسافة (L2 Distance)
query2 = "Which vector database is built with Rust?"
faiss_results = faiss_store.similarity_search_with_score(query2, k=2)

print("🎯 نتائج FAISS مع المسافات (أقل = أكثر شبهاً):")
for doc, score in faiss_results:
    print(f"• المسافة: {score:.4f} | المحتوى: {doc.page_content[:120]}...")

# 3. حفظ الفهرس على القرص وإعادة تحميله
faiss_dir = DATA_DIR / "faiss_test_index"
os.makedirs(faiss_dir, exist_ok=True)
faiss_store.save_local(str(faiss_dir))

loaded_faiss = FAISS.load_local(str(faiss_dir), hf_embeddings, allow_dangerous_deserialization=True)
print(f"
💾 تم حفظ وإعادة تحميل فهرس FAISS بنجاح من: {faiss_dir.resolve()}")


<div dir="rtl">

## 5️⃣ النوع الثالث: `SKLearnVectorStore` (Scikit-Learn KNN & JSON Serializer)
يعتمد على خوارزميات `NearestNeighbors` في Scikit-Learn مع إمكانية تصدير الفهرس كـ JSON.

</div>


In [ ]:
from langchain_community.vectorstores import SKLearnVectorStore

# مسار حفظ الفهرس بصيغة JSON
sklearn_json = DATA_DIR / "sklearn_test.json"

# بناء مستودع Scikit-Learn مع تحديد مسار الحفظ المسبق
sklearn_store = SKLearnVectorStore.from_documents(
    documents=docs,
    embedding=hf_embeddings,
    persist_path=str(sklearn_json),
    serializer="json",
    algorithm="brute",
    metric="cosine"
)

# البحث
query3 = "How does Cosine Similarity differ from Euclidean distance?"
sklearn_res = sklearn_store.similarity_search(query3, k=2)

print("🎯 نتائج SKLearnVectorStore:")
for i, r in enumerate(sklearn_res, 1):
    print(f"{i}. {r.page_content[:130]}...\n")

# حفظ الفهرس على القرص
sklearn_store.persist()
print(f"💾 تم حفظ فهرس Scikit-Learn بصيغة JSON في: {sklearn_json.resolve()}")


<div dir="rtl">

## 6️⃣ النوع الرابع: `DocArrayInMemorySearch` (Ultra-Fast Memory Store)
مستودع في الذاكرة فائق السرعة والخفة.

</div>


In [ ]:
from langchain_community.vectorstores import DocArrayInMemorySearch

docarray_store = DocArrayInMemorySearch.from_documents(docs, hf_embeddings)
docarray_res = docarray_store.similarity_search_with_score("ArxivLoader research papers", k=2)

print("🎯 نتائج DocArrayInMemorySearch:")
for doc, score in docarray_res:
    print(f"• الدرجة: {score:.4f} | المحتوى: {doc.page_content[:120]}...")


<div dir="rtl">

## 7️⃣ النوع الخامس والسادس: `Chroma` و `Qdrant`
المستودعات المخصصة للبيئات الكبيرة والإنتاجية (Chroma DB & Qdrant Engine).

</div>


In [ ]:
# تجربة Chroma
try:
    from langchain_chroma import Chroma
    chroma_store = Chroma.from_documents(
        docs, 
        embedding=hf_embeddings,
        collection_name="test_collection"
    )
    chroma_res = chroma_store.similarity_search("Chroma DB SQLite persistence", k=1)
    print("✅ Chroma DB يعمل بنجاح:", chroma_res[0].page_content[:100])
except Exception as e:
    print(f"ℹ️ تنبيه بيئة التشغيل لـ Chroma DB: {e}")

# تجربة Qdrant
try:
    from langchain_qdrant import QdrantVectorStore
    qdrant_store = QdrantVectorStore.from_documents(
        docs,
        embedding=hf_embeddings,
        location=":memory:",
        collection_name="test_qdrant"
    )
    qdrant_res = qdrant_store.similarity_search("Qdrant Rust concurrency", k=1)
    print("✅ Qdrant يعمل بنجاح:", qdrant_res[0].page_content[:100])
except Exception as e:
    print(f"ℹ️ تنبيه بيئة التشغيل لـ Qdrant: {e}")


<div dir="rtl">

## 8️⃣ اختبار استراتيجيات الاسترجاع (Retrievers & Search Types)
مقارنة نتائج البحث بالتشابه القياسي (`similarity`) مع البحث بالتنوع الأقصى (**`MMR`**) لتفادي تكرار المقاطع.

</div>


In [ ]:
test_query = "What are the text splitting strategies and chunking best practices?"

# 1. استرجاع بالتشابه القياسي
sim_retriever = faiss_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})
sim_docs = sim_retriever.invoke(test_query)

# 2. استرجاع مع MMR للتنوع
mmr_retriever = faiss_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 6, "lambda_mult": 0.5}
)
mmr_docs = mmr_retriever.invoke(test_query)

print("🔎 [Similarity Retriever]:")
for d in sim_docs:
    print(f"  • [Chunk #{d.metadata.get('chunk_id')}]: {d.page_content[:100]}...")

print("
🌟 [MMR Retriever (تنوع وتقليل التكرار)]:")
for d in mmr_docs:
    print(f"  • [Chunk #{d.metadata.get('chunk_id')}]: {d.page_content[:100]}...")


<div dir="rtl">

## 9️⃣ بناء واختبار سلسلة الـ RAG الكاملة عبر LCEL
توصيل المسترجع مع قالب Prompt للحصول على إجابات موثقة بالأدلة.

</div>


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
    return "

".join([f"[Chunk #{d.metadata.get('chunk_id')}]: {d.page_content}" for d in retrieved_docs])

template = """أنت خبير ذكاء اصطناعي. أجب عن السؤال التالي بالاعتماد فقط على السياق المرفق مع ذكر أرقام المقاطع المستخدمة:

السياق:
{context}

السؤال: {question}

الإجابة الموثقة:"""

prompt = PromptTemplate.from_template(template)

# تكوين السلسلة
rag_pipeline = (
    {"context": mmr_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | StrOutputParser()
)

# اختبار السلسلة
output_prompt = rag_pipeline.invoke("ما هي أحجام التقسيم الموصى بها للـ Q&A؟")
print("📋 ناتج السلسلة المهيأة للإرسال إلى الـ LLM:
")
print(output_prompt)
